In [0]:
spark.sql("USE CATALOG iitb")
spark.sql("USE SCHEMA bharat_bricks")

In [0]:
posts_path = "/Volumes/iitb/bharat_bricks/data/stations.json"



In [0]:
from pyspark.sql.functions import col, explode

# Load the data from JSON
df = spark.read.json(posts_path)

# Display schema to understand structure
print(f"\nSchema:")
df.printSchema()

# Explode features to show individual records
df_features = df.select(explode(col("features")).alias("feature"))
print(f"\nTotal records: {df_features.count()}")

# Show top 10 records
print("\nShowing top 10 records:")
display(df_features.limit(10))

In [0]:
from pyspark.sql.functions import col, explode, collect_list, struct, lit

# Explode the features array to work with individual records
df_exploded = df.select(explode(col("features")).alias("feature"))

# Count total records before cleaning
total_records = df_exploded.count()
print(f"Total records before cleaning: {total_records}")

# Filter out records where geometry is null
df_clean = df_exploded.filter(col("feature.geometry").isNotNull())

# Count records after cleaning
clean_records = df_clean.count()
null_records = total_records - clean_records

print(f"Records with null geometry (removed): {null_records}")
print(f"Clean records (kept): {clean_records}")
print(f"\nPercentage of data retained: {(clean_records/total_records)*100:.2f}%")

# Show sample of clean data
display(df_clean.limit(10))

In [0]:
# Reconstruct the GeoJSON structure with cleaned features
df_cleaned_geojson = df_clean.select(col("feature")).groupBy().agg(
    collect_list("feature").alias("features")
).withColumn("type", lit("FeatureCollection"))

# Define output path for cleaned data
cleaned_path = "/Volumes/iitb/bharat_bricks/data/stations_cleaned.json"

# Save the cleaned data
df_cleaned_geojson.coalesce(1).write.mode("overwrite").json(cleaned_path)

print(f"✓ Cleaned data saved to: {cleaned_path}")
print(f"✓ Removed {null_records} records with null geometry")
print(f"✓ Retained {clean_records} valid records")

In [0]:
posts_path = "/Volumes/iitb/bharat_bricks/data/schedules.json"


In [0]:
# Load schedules data
df_schedules = spark.read.json(posts_path)

print("Schema:")
df_schedules.printSchema()

print(f"\nTotal records: {df_schedules.count()}")

# Show sample data
print("\nSample data (top 10 records):")
display(df_schedules.limit(10))